# Docling · loader moderno de IBM

Docling (IBM, 2024+) es un **document loader moderno** que entiende layout, tablas, fórmulas e imágenes. Donde `PyPDFLoader` te da el texto crudo (a veces mal ordenado por columnas), Docling te da una estructura.

## ¿Cuándo usar Docling vs PyPDFLoader?

| Caso | Loader recomendado |
|---|---|
| PDF de texto plano (papers, reports) | `PyPDFLoader` (más rápido) |
| PDF con tablas críticas | **Docling** |
| PDF escaneado (necesita OCR) | **Docling** (con OCR built-in) |
| DOCX, PPTX, XLSX, HTML, MD unificado | **Docling** |
| Layout multi-columna | **Docling** |

## Stack 2026

- `docling>=2` · API estable
- Pipeline VLM opcional con `GraniteDocling` (258M, sucesor de SmolDocling) para casos avanzados

## Setup

In [1]:
%pip install -q "docling>=2" "langchain-core>=0.3"

## 1. Conversión simple · arxiv paper

Le pasamos una URL de arxiv y Docling se ocupa de descargar, parsear y devolver un objeto estructurado.

In [2]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
url = "https://arxiv.org/pdf/2408.09869"  # Docling technical report
result = converter.convert(url)

print(f"  ✓ convertido")
print(f"  páginas: {len(result.document.pages)}")  # nota: pages es dict {page_no: PageItem}
# `document.name` es el nombre del fichero ("2408"); el título real es el primer encabezado detectado
from docling_core.types.doc import DocItemLabel
titulo = next((t.text for t in result.document.texts
               if t.label in (DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER)), result.document.name)
print(f"  título: {titulo}")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 9501.74it/s]

[INFO] 2026-10-14 17:25:03,226 [RapidOCR] base.py:23: Using engine_name: onnxruntime


[INFO] 2026-10-14 17:25:03,427 [RapidOCR] base.py:23: Using engine_name: onnxruntime


[INFO] 2026-10-14 17:25:03,454 [RapidOCR] base.py:23: Using engine_name: onnxruntime


RapidOCR returned empty result!


  ✓ convertido
  páginas: 9
  título: Docling Technical Report


## 2. Exportar a Markdown

Docling convierte a markdown estructurado, respetando jerarquía de encabezados, tablas y listas. Esto es **directamente** lo que quieres pasar a un chunker markdown-aware.

In [3]:
md_text = result.document.export_to_markdown()

# Mostramos primer trozo
print(md_text[:1500])
print(f"\n... (total {len(md_text)} caracteres)")

<!-- image -->

## Docling Technical Report

## Version 1.0

Christoph Auer MaksymLysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla LokeshMishra Yusik Kim ShubhamGupta Rafael Teixeira de Lima ValeryWeber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk PeterW. J. Staar

AI4KGroup, IBMResearch R¨uschlikon, Switzerland

## Abstract

This technical report introduces Docling, an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.

## 1 Introduction

Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability i

## 3. Tablas · Docling las preserva

Donde un PDF reader tradicional convierte tablas en una sopa de palabras desordenadas, Docling te da la tabla **estructurada** que puedes inspeccionar.

In [4]:
tables = result.document.tables if hasattr(result.document, "tables") else []
print(f"  tablas detectadas: {len(tables)}")

if tables:
    # Mostramos la primera tabla
    first = tables[0]
    print("\nPrimera tabla (primeras filas):")
    df = first.export_to_dataframe(doc=result.document)
    print(df.head())

  tablas detectadas: 3

Primera tabla (primeras filas):
                                          CPU. Thread budget.  \
0                                 Apple M3 Max           4 16   
1  (16 cores) Intel(R) Xeon E5-2690 (16 cores)           4 16   

  native backend.TTS native backend.Pages/s native backend.Mem  \
0        177 s 167 s              1.27 1.34            6.20 GB   
1        375 s 244 s              0.60 0.92            6.16 GB   

  pypdfium backend.TTS pypdfium backend.Pages/s pypdfium backend.Mem  
0           103 s 92 s                2.18 2.45              2.56 GB  
1          239 s 143 s                0.94 1.57              2.42 GB  


## 4. Chunking nativo · `HybridChunker`

Docling incluye su propio chunker que aprovecha la estructura. Respeta secciones, mantiene tablas en un solo chunk, y enriquece cada chunk con su contexto jerárquico (qué sección, qué subsección).

In [5]:
from docling.chunking import HybridChunker

chunker = HybridChunker()
chunks = list(chunker.chunk(dl_doc=result.document))
print(f"  ✓ {len(chunks)} chunks")

# Elegimos el primer chunk de una sección numerada ("1 Introduction"...) para ver su contexto jerárquico
ejemplo = next((c for c in chunks if c.meta.headings and c.meta.headings[-1][:1].isdigit()), chunks[0])
print(f"\nEjemplo de chunk:")
print(f"  headings (sección > subsección): {ejemplo.meta.headings}")
print(f"  text: {ejemplo.text[:300]}...")

# contextualize() antepone los encabezados al texto: es lo que conviene embeber
print(f"\nMismo chunk contextualizado (lo que pasarías al embedding):")
print(chunker.contextualize(chunk=ejemplo)[:400], "...")

# Las tablas no se trocean a mitad: cada chunk de tabla contiene solo filas de tabla
from docling_core.types.doc import DocItemLabel
n_tabla = sum(all(d.label == DocItemLabel.TABLE for d in c.meta.doc_items) for c in chunks)
print(f"\nChunks que son (solo) tabla: {n_tabla}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (707 > 512). Running this sequence through the model will result in indexing errors


  ✓ 48 chunks

Ejemplo de chunk:
  headings (sección > subsección): ['1 Introduction']
  text: Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimized characteristic, which discards most structural features and metadata. With the advent of LLMs and popular appli...

Mismo chunk contextualizado (lo que pasarías al embedding):
1 Introduction
Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variability in formats, weak standardization and printing-optimized characteristic, which discards most structural features and metadata. With the advent of LLMs and popular application patterns such as retrieval-augmented generation (RAG), leveraging the rich con ...

Chunks que son (solo) tabla: 7


## 5. Conversión a Documents de Langchain

Para integrar con un pipeline RAG existente, basta con envolver cada chunk de Docling en un `Document` de Langchain.

In [6]:
from langchain_core.documents import Document

lc_docs = []
for c in chunks:
    metadata = {"source": url}
    if hasattr(c, "meta") and c.meta:
        # añadimos contexto jerárquico (qué sección/subsección)
        try:
            metadata["headings"] = " > ".join(c.meta.headings or [])
        except Exception:
            pass
    lc_docs.append(Document(page_content=c.text, metadata=metadata))

print(f"  ✓ {len(lc_docs)} documents listos para indexar en cualquier vectorstore")
print(f"\nEjemplo:")
print(f"  metadata: {lc_docs[0].metadata}")
print(f"  text[:200]: {lc_docs[0].page_content[:200]}...")

  ✓ 48 documents listos para indexar en cualquier vectorstore

Ejemplo:
  metadata: {'source': 'https://arxiv.org/pdf/2408.09869', 'headings': 'Version 1.0'}
  text[:200]: Christoph Auer MaksymLysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla LokeshMishra Yusik Kim ShubhamGupta Rafael ...


## 6. Multi-formato · DOCX, PPTX, HTML

El mismo `DocumentConverter` acepta también DOCX, PPTX, XLSX, HTML, imágenes (con OCR), y más. Una sola interfaz para todos los formatos típicos de empresa.

In [7]:
# Ejemplo: si tuvieras un .docx local
# result_docx = converter.convert("./reporte.docx")
# md_text = result_docx.document.export_to_markdown()

print("Formatos soportados (resumen):")
print("  - PDF (con OCR opcional)")
print("  - DOCX, PPTX, XLSX")
print("  - HTML")
print("  - Markdown")
print("  - Imágenes (PNG, JPG con OCR)")
print("  - URLs (descarga + parseo)")

Formatos soportados (resumen):
  - PDF (con OCR opcional)
  - DOCX, PPTX, XLSX
  - HTML
  - Markdown
  - Imágenes (PNG, JPG con OCR)
  - URLs (descarga + parseo)


## Cierre

- Docling te da una **estructura**, no solo texto crudo
- Tablas, encabezados y listas se preservan correctamente
- El `HybridChunker` aprovecha esa estructura para chunks más limpios
- Se integra con Langchain en 5 líneas
- **Cuándo usarlo**: cuando los PDFs tienen tablas que importan o layout no trivial. Para texto plano, `PyPDFLoader` es más rápido.